# 05 · Validation plan — pNP-ester kinetics, controls, enantioselectivity

**Standard slot:** *validation plan.* **For Project 21 this means:** turn the catalytic-geometry-
filtered set into a costed **pNP-ester steady-state kinetics + DSF plan** with the right controls
(incl. a **catalytic-Ser→Ala dead mutant**) and an **enantioselectivity / kinetic-resolution** concept
for hits (D4/D5).

This is the deliverable that states, plainly: **in-silico geometry is a hypothesis; the assay tests it.**

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Select the synthesis set (< 96 designs)
Pick the top designs from the ranked filter (catalytic geometry first, then an accessible pocket),
capped at **< 96** so they fit a single screening plate with controls. Diversity matters — don't pick
96 near-identical designs; spread across scaffolds and (where relevant) acyl-chain preferences.

In [ ]:
import pandas as pd
try:
    ranked = pd.read_csv("results/ranked.csv")
except FileNotFoundError:
    ranked = pd.read_csv("results/campaign.csv")

# Prefer designs passing the catalytic-geometry bar; cap under 96 (leave wells for controls).
ok = ranked[ranked["catalytic_geom_rmsd"] <= 0.5] if "catalytic_geom_rmsd" in ranked else ranked
selected = ok.head(88).copy()
selected.to_csv("results/synthesis_set.csv", index=False)
print(f"selected {len(selected)} designs for synthesis (< 96, leaving wells for controls) [SYNTHETIC ranking]")
print("Diversify across scaffolds + acyl-chain preferences; record why each was chosen in your report.")

## 2 · The kinetic assay (pNP-ester product readout) + DSF
- **Express + purify:** *E. coli* BL21(DE3), 16–18 °C overnight; His-tag → IMAC → SEC polish.
- **Assay:** follow **p-nitrophenolate** release from p-nitrophenyl acetate (pNPA, C2) or butyrate
  (pNPB, C4) by absorbance at **~405–410 nm** in a plate reader; take initial rates across substrate
  concentrations.
- **Kinetics:** fit **kcat** and **KM** (and kcat/KM) from a Michaelis–Menten / linear-regime fit.
- **DSF:** a thermal-shift (DSF) melt for each construct — confirms the fold and flags unstable designs.
- **Substrate scope:** repeat the assay across the acyl-chain series (C2/C4/C6/C8 pNP-esters) to map
  esterase- vs lipase-like preference.
- **Readout note:** background (non-enzymatic) pNP-ester hydrolysis is non-zero — subtract the
  buffer-only rate, and account for it in every well.

In [ ]:
controls = {
    "POSITIVE — natural/reference serine hydrolase": "a verified cutinase/lipase; confirms the assay works",
    "NEGATIVE — catalytic-Ser -> Ala 'dead' mutant": "SAME design, catalytic serine mutated to Ala; the "
        "cleanest negative — loss of activity pins catalysis to that residue (use make_dead_mutant)",
    "NEGATIVE — empty-vector lysate": "no insert; rules out host-background esterase activity",
    "BLANK — buffer + substrate only": "the non-enzymatic background pNP-ester rate to subtract",
}
print("MANDATORY controls (every plate):")
for k, v in controls.items():
    print(f"  - {k}\n      {v}")

# Show the dead-mutant control is a one-residue edit of a selected design (SYNTHETIC sequence).
from enzyme_tools import make_dead_mutant
sel = pd.read_csv("results/synthesis_set.csv")
if "sequence" in sel and len(sel):
    seq = str(sel.iloc[0]["sequence"]); ser_idx = seq.find("S")
    if ser_idx >= 0:
        dead = make_dead_mutant(seq, ser_idx)
        n_diff = sum(a != b for a, b in zip(seq, dead))
        print(f"\ndead-mutant control for {sel.iloc[0]['design_id']}: S{ser_idx}A; differs at "
              f"exactly {n_diff} position (same fold, no nucleophile).")

## 3 · A costed, plate-based screen (template — fill real prices)
One 96-well plate holds the < 96 designs + the controls above. Cost the gene synthesis, expression,
and assay reagents at your institution's rates; the cell prints a template to fill in your report.

In [ ]:
plan = [
    ("Gene synthesis (codon-optimised, screened provider)", "< 96 designs", "fill price/construct"),
    ("Cloning + transformation", "1 plate", "fill"),
    ("Expression + lysis", "1 plate", "fill"),
    ("IMAC purification (plate format)", "1 plate", "fill"),
    ("p-nitrophenyl ester substrates (C2/C4/C6/C8)", "stock", "fill"),
    ("DSF dye + plate-reader time", "per plate", "fill"),
    ("Plate-reader time (kinetics, 405 nm)", "per plate", "fill"),
]
print("Costed reagent/step list (fill institutional prices) [TEMPLATE]:")
for step, scale, cost in plan:
    print(f"  - {step:54s} {scale:14s} {cost}")
print("\nTimeline (typical): synthesis 2-3 wk -> clone/express 1-2 wk -> purify+assay 1-2 wk.")
print("Synthesis MUST go through an IGSC-member, biosecurity-screening provider (low dual-use here, "
      "but it is policy). Wet-lab needs institutional biosafety/ethics sign-off.")

## 4 · Enantioselectivity / kinetic resolution `[stretch]`
The green-chemistry payoff of a designed esterase is often **kinetic resolution** of a racemic ester:
- **Readout:** assay a **chiral pNP-ester** (or follow each enantiomer by chiral HPLC/GC); compute the
  **enantiomeric ratio E** from the relative kcat/KM of the two enantiomers and the ee vs conversion.
- **Design link:** face selectivity is set by the **asymmetric pocket** around the acyl/leaving groups —
  relate measured E to the pocket features your LigandMPNN design installed.
- **Honest framing:** a high E is hard and usually needs iteration; report the E-value trajectory, not
  a single best clone. (Concept only at this stage — no fabricated ee.)

In [ ]:
print("Enantioselectivity (stretch): assay a CHIRAL pNP-ester -> initial rates per enantiomer ->")
print("compute the enantiomeric ratio E and ee vs conversion; relate face selectivity to the pocket.")
print("Reminder for the thesis: report the hit rate and the kcat/KM (and E) DISTRIBUTION, not just the")
print("best clone. NO kcat/KM/ee is fabricated here — these come from the wet-lab assay.")

## D4 / D5 checklist
- [ ] `results/synthesis_set.csv`: < 96 diverse, catalytic-geometry-passing, pocket-accessible designs.
- [ ] pNP-ester kinetic-assay + DSF plan: 405 nm product readout, kcat/KM, with **all** controls
      (catalytic-Ser→Ala dead mutant, natural reference, empty vector, blank), costed + timed.
- [ ] Substrate-scope panel (acyl-chain series) in the plan.
- [ ] Enantioselectivity / kinetic-resolution concept `[stretch]`.
- [ ] Thesis chapter + 15-min talk + `v1.0` tagged release; "geometry ≠ activity" stated plainly.

You're done — and this project reuses the Project 18/19 theozyme→scaffold→sequence→geometry template,
swapping in ester hydrolysis (triad + oxyanion hole) and the pocket→scope→kinetics emphasis.